# Explore the macroalgal microbiome use case

This notebook analyzes MAGs for the macroalgal microbiome use case.

**Inputs**: data in `../data/marine-use-case/data/`:
- `drep.csv`: dRep clustering output
- `checkm2.tsv`: CheckM2 quality assessment
- `gtdb.tsv`: GTDB taxonomy classification
- `quast.tsv`: QUAST assembly statistics
- `bakta.tsv`: BAKTA genome annotation

**Outputs**
- Plots: `../results/marine-use-case/`

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

sys.path.insert(0, str((Path.cwd() / "bin").resolve()))
sys.path.insert(0, str(Path.cwd().resolve()))

from helpers import (
    load_dfs,
    compute_print_stats,
    explore_species_level_clusters_all,
    compute_taxo_classification_summary,
    get_all_taxo_levels,
    get_bakta_annot_df,
    get_kegg_path_df,
    get_relative_abund_taxo_levels,
    print_stats,
)

In [2]:
uc_name = "macroalgal-epiphytic"

metadata_df, reps_df, coverage_df = load_dfs(uc_name)

# Quality

In [3]:
print(f"Total number of MAGs: {reps_df['Cluster members'].sum()}")
print(f"Total number of species-level clusters: {reps_df.shape[0]}")

Total number of MAGs: 897
Total number of species-level clusters: 832


In [4]:
explore_species_level_clusters_all(reps_df)

Species-level clusters with no contamination threshold
Total number: 832.0
Cluster members: 1.08 ± 0.30, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-3.00
Contamination: 1.42 ± 1.95, Median: 0.62, IQR: 0.06-1.91, Range: 0.00-10.18
Completeness: 44.39 ± 36.55, Median: 30.80, IQR: 9.70-88.40, Range: 0.00-100.00
Total length: 1.82 ± 1.61, Median: 1.28, IQR: 0.40-2.96, Range: 0.05-7.59



In [5]:
mapped = coverage_df.query("Genome != 'unmapped'").drop(columns="Genome").sum(axis=0)
unmapped = 100 - mapped
print_stats(unmapped.describe().to_frame("Unmapped reads"))
print_stats(mapped.describe().to_frame("Mapped reads (sum over all genomes, stats over samples)"))

Unmapped reads: 54.92 ± 16.42, Median: 50.71, IQR: 45.86-61.87, Range: 41.01-73.04
Mapped reads (sum over all genomes, stats over samples): 45.08 ± 16.42, Median: 49.29, IQR: 38.13-54.14, Range: 26.96-58.99


# Clusters given Bowers et al / MIMAG classification

## HQ: High-quality species-level clusters (contamination < 5% and completeness > 90%)

In [6]:
hq_df = reps_df.query("Contamination < 5 and Completeness > 90")
hq_df

,MAG,Domain,Phylum,Class,Order,Family,Genus,Species,Domain (GTDB),Phylum (GTDB),...,kegg_dTDP-D-angolosamine biosynthesis,kegg_dTDP-D-desosamine biosynthesis,kegg_dTDP-D-forosamine biosynthesis,kegg_dTDP-D-mycaminose biosynthesis,kegg_dTDP-L-megosamine biosynthesis,kegg_dTDP-L-mycarose biosynthesis,kegg_dTDP-L-oleandrose biosynthesis,kegg_dTDP-L-olivose biosynthesis,kegg_dTDP-L-rhamnose biosynthesis,kegg_dTDP-beta-L-noviose biosynthesis
0,SRR22878281_binette_bin2,Bacteria,Pseudomonadota,Alphaproteobacteria,Rhizobiales,Hyphomicrobiaceae,CAJQQK01,unclassified,Bacteria,Pseudomonadota,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
1,SRR22878283_binette_bin9,Bacteria,Bdellovibrionota,Bacteriovoracia,Bacteriovoracales,Bacteriovoracaceae,CAKZJW01,unclassified,Bacteria,Bdellovibrionota,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,NaN
2,SRR22878281_binette_bin11,Bacteria,Pseudomonadota,Gammaproteobacteria,Arenicellales,Arenicellaceae,Arenicella,unclassified,Bacteria,Pseudomonadota,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
3,SRR22878281_binette_bin11,Bacteria,Pseudomonadota,Gammaproteobacteria,Arenicellales,Arenicellaceae,Arenicella,unclassified,Bacteria,Pseudomonadota,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
4,SRR22878281_binette_bin12,Bacteria,Pseudomonadota,Gammaproteobacteria,Granulosicoccales,Granulosicoccaceae,Granulosicoccus,unclassified,Bacteria,Pseudomonadota,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,SRR22878281_binette_bin45,Bacteria,Pseudomonadota,Alphaproteobacteria,Caulobacterales,Maricaulaceae,JACOMS01,unclassified,Bacteria,Pseudomonadota,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
186,SRR22878281_binette_bin45,Bacteria,Pseudomonadota,Alphaproteobacteria,Caulobacterales,Maricaulaceae,JACOMS01,unclassified,Bacteria,Pseudomonadota,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
187,SRR22878283_binette_bin36,Bacteria,Bacteroidota,Bacteroidia,Flavobacteriales,Flavobacteriaceae,Croceitalea,Croceitalea vernalis,Bacteria,Bacteroidota,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
188,SRR22878282_binette_bin21,Bacteria,Bacteroidota,Bacteroidia,Flavobacteriales,Flavobacteriaceae,Postechiella,unclassified,Bacteria,Bacteroidota,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,NaN


In [7]:
compute_print_stats(hq_df) 
print(f"Total number: {len(hq_df)} ({len(hq_df) / len(reps_df) * 100:.2f}%)")

Total number: 182.0
Cluster members: 1.18 ± 0.46, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-3.00
Contamination: 1.19 ± 1.05, Median: 1.02, IQR: 0.27-1.72, Range: 0.00-4.65
Completeness: 96.31 ± 3.25, Median: 96.90, IQR: 93.28-99.60, Range: 90.10-100.00
Total length: 3.76 ± 1.17, Median: 3.55, IQR: 2.96-4.43, Range: 1.03-7.12
Total number: 182 (21.88%)


### Taxonomy

In [8]:
compute_taxo_classification_summary(hq_df)

,Unclassified clusters,Classified clusters,Unclassified clusters %,Classified clusters %
Domain,0,182,0.0,100.0
Phylum,0,182,0.0,100.0
Class,0,182,0.0,100.0
Order,0,182,0.0,100.0
Family,6,176,3.3,96.7
Genus,24,158,13.19,86.81
Species,168,14,92.31,7.69


In [9]:
hq_taxo_levels = get_all_taxo_levels(hq_df)


Level: Domain


,Cluster,Cluster %,Total MAG count
Domain,,,
Bacteria,182.0,100.0,215.0
TOTAL,182.0,100.0,215.0



Level: Phylum


,Cluster,Cluster %,Total MAG count
Phylum,,,
Pseudomonadota,104.0,57.142857,112.0
Bacteroidota,40.0,21.978022,61.0
Cyanobacteriota,10.0,5.494505,10.0
Actinomycetota,6.0,3.296703,6.0
Myxococcota_A,6.0,3.296703,6.0
Bdellovibrionota,4.0,2.197802,6.0
Patescibacteriota,4.0,2.197802,6.0
Acidobacteriota,2.0,1.098901,2.0
Bdellovibrionota_B,2.0,1.098901,2.0



Level: Class


...

,Cluster,Cluster %,Total MAG count
Family,,,
Flavobacteriaceae,34.0,18.681319,55.0
Maricaulaceae,14.0,7.692308,14.0
Cellvibrionaceae,12.0,6.593407,13.0
Alteromonadaceae,8.0,4.395604,13.0
Granulosicoccaceae,12.0,6.593407,12.0
Sphingomonadaceae,11.0,6.043956,11.0
Xenococcaceae,8.0,4.395604,8.0
Arenicellaceae,7.0,3.846154,7.0
Geminicoccaceae,6.0,3.296703,6.0



Level: Genus


,Cluster,Cluster %,Total MAG count
Genus,,,
unclassified,24.0,13.186813,26.0
Aquimarina,10.0,5.494505,12.0
Postechiella,4.0,2.197802,10.0
Arenicella,7.0,3.846154,7.0
Maribacter_C,4.0,2.197802,6.0
...,...,...,...
Agarilytica,2.0,1.098901,2.0
Lacinutrix,1.0,0.549451,2.0
Psychrobacter,1.0,0.549451,1.0



Level: Species


,Cluster,Cluster %,Total MAG count
Species,,,
unclassified,168.0,92.307692,188.0
Cellulophaga lytica,1.0,0.549451,3.0
Dokdonia sp947496725,1.0,0.549451,3.0
Olleya sediminilitoris,1.0,0.549451,3.0
Pseudoalteromonas marina,1.0,0.549451,3.0
Croceitalea vernalis,1.0,0.549451,2.0
Lacinutrix sp000211855,1.0,0.549451,2.0
Maribacter litoralis,1.0,0.549451,2.0
Marinagarivorans sp947494095,1.0,0.549451,2.0


### Relative abundance

In [10]:
hq_relative_abund_df = get_relative_abund_taxo_levels(hq_df, coverage_df)

Unmapped reads: 80.68 ± 7.93, Median: 77.16, IQR: 76.14-83.46, Range: 75.12-89.76
Mapped reads: 19.32 ± 7.93, Median: 22.84, IQR: 16.54-23.86, Range: 10.24-24.88

Level: Family


,count,mean,std,min,25%,50%,75%,max
Family,,,,,,,,
Flavobacteriaceae,3.0,45.917115,11.086574,36.242300,39.868309,43.494317,50.754522,58.014728
Cellvibrionaceae,3.0,12.370510,17.156165,0.326603,2.548839,4.771075,18.392464,32.013853
Sphingomonadaceae,3.0,5.286303,4.406424,0.234318,3.761265,7.288212,7.812295,8.336379
Alteromonadaceae,3.0,4.995996,3.357696,2.053601,3.167139,4.280677,6.467194,8.653711
Maricaulaceae,3.0,3.494973,2.878067,0.404811,2.192941,3.981072,5.040054,6.099036
Xenococcaceae,3.0,2.928851,3.484666,0.039074,0.993993,1.948913,4.373740,6.798567
Granulosicoccaceae,3.0,2.833944,2.185364,0.374142,1.975146,3.576150,4.063845,4.551540
Arenicellaceae,3.0,2.217070,2.607447,0.128755,0.755844,1.382933,3.261227,5.139521
Hyphomonadaceae,3.0,1.969531,2.629104,0.083174,0.467962,0.852750,2.912709,4.972668



Level: Genus


,count,mean,std,min,25%,50%,75%,max
Genus,,,,,,,,
unclassified,3.0,16.864858,20.957848,4.155741,4.770038,5.384335,23.219417,41.054499
Aquimarina,3.0,15.481835,23.306296,0.200958,2.069242,3.937526,23.122273,42.307020
Cellulophaga,3.0,10.178136,13.866727,0.301126,2.251786,4.202447,15.116640,26.030834
Postechiella,3.0,6.168215,6.186407,0.690269,2.813425,4.936581,8.907188,12.877794
Dokdonia,3.0,4.216697,4.807290,1.005900,1.453250,1.900600,5.822095,9.743590
...,...,...,...,...,...,...,...,...
CAKZUP01,3.0,0.196244,0.301589,0.000000,0.022612,0.045223,0.294366,0.543510
CALHVX01,3.0,0.186574,0.268755,0.012306,0.031818,0.051329,0.273708,0.496087
JAUIGD01,3.0,0.169676,0.293887,0.000000,0.000000,0.000000,0.254514,0.509027



Level: Species


,count,mean,std,min,25%,50%,75%,max
Species,,,,,,,,
unclassified,3.0,78.459246,18.961977,58.060492,69.914462,81.768433,88.658622,95.548812
Cellulophaga lytica,3.0,10.178136,13.866727,0.301126,2.251786,4.202447,15.116640,26.030834
Dokdonia sp947496725,3.0,4.216697,4.807290,1.005900,1.453250,1.900600,5.822095,9.743590
Pseudoalteromonas atlantica,3.0,1.413222,1.257358,0.157823,0.783568,1.409312,2.040922,2.672531
Lacinutrix sp000211855,3.0,1.361587,1.167712,0.129317,0.816515,1.503712,1.977722,2.451731
Maribacter litoralis,3.0,1.104303,1.363866,0.086823,0.329433,0.572042,1.613043,2.654043
Pseudoalteromonas marina,3.0,0.905768,0.924540,0.334394,0.372437,0.410479,1.191454,1.972430
Marinagarivorans sp947494095,3.0,0.423268,0.481044,0.021210,0.156798,0.292387,0.624298,0.956208
Olleya sediminilitoris,3.0,0.417397,0.216793,0.201522,0.308547,0.415572,0.525334,0.635096


### Functions

In [11]:
print_stats(get_bakta_annot_df(hq_df).describe())

CDSs: 3374.03 ± 945.84, Median: 3211.00, IQR: 2693.50-4055.75, Range: 1037.00-5968.00
CRISPR arrays: 0.21 ± 0.77, Median: 0.00, IQR: 0.00-0.00, Range: 0.00-4.00
gaps: 0.00 ± 0.00, Median: 0.00, IQR: 0.00-0.00, Range: 0.00-0.00
hypotheticals: 1032.20 ± 594.48, Median: 904.50, IQR: 564.00-1365.50, Range: 184.00-3018.00
ncRNA regions: 5.92 ± 2.82, Median: 6.00, IQR: 4.00-8.00, Range: 1.00-15.00
ncRNAs: 6.04 ± 6.52, Median: 4.00, IQR: 3.00-6.00, Range: 1.00-51.00
oriCs: 0.03 ± 0.19, Median: 0.00, IQR: 0.00-0.00, Range: 0.00-2.00
oriTs: 0.00 ± 0.00, Median: 0.00, IQR: 0.00-0.00, Range: 0.00-0.00
oriVs: 0.00 ± 0.00, Median: 0.00, IQR: 0.00-0.00, Range: 0.00-0.00
pseudogenes: 5.20 ± 6.15, Median: 3.00, IQR: 1.00-7.00, Range: 0.00-33.00
rRNAs: 1.15 ± 1.19, Median: 1.00, IQR: 0.00-2.00, Range: 0.00-8.00
sORFs: 0.23 ± 0.46, Median: 0.00, IQR: 0.00-0.00, Range: 0.00-2.00
signal peptides: 0.00 ± 0.00, Median: 0.00, IQR: 0.00-0.00, Range: 0.00-0.00
tRNAs: 36.68 ± 8.60, Median: 35.00, IQR: 30.00-42.

In [12]:
get_kegg_path_df(hq_df)

Before removing rows and columns with only zeros:
Clusters: 182
KEGG modules: 405

After removing rows and columns with only zeros:
Clusters: 182
KEGG modules: 378

KEGG modules: 194.59 ± 21.95, Median: 195.00, IQR: 186.25-205.00, Range: 78.00-245.00


,"10-membered enediyne core biosynthesis, malonyl-CoA => 3-hydroxyhexadeca-4,6,8,10,12,14-hexaenoyl-ACP => 10-membered enediyne core",3-Hydroxypropionate bi-cycle,"9-membered enediyne core biosynthesis, malonyl-CoA => 3-hydroxyhexadeca-4,6,8,10,12,14-hexaenoyl-ACP => 9-membered enediyne core",ADP-L-glycero-D-manno-heptose biosynthesis,"Abscisic acid biosynthesis, beta-carotene => abscisic acid","Acarbose biosynthesis, sedoheptulopyranose-7P => acarbose",Acylglycerol degradation,"Adenine ribonucleotide biosynthesis, IMP => ADP,ATP","Adenine ribonucleotide degradation, AMP => Urate","Aerobactin biosynthesis, lysine => aerobactin",...,beta-Oxidation,"beta-Oxidation, acyl-CoA synthesis","beta-Oxidation, peroxisome, VLCFA","beta-Oxidation, peroxisome, tri/dihydroxycholestanoyl-CoA => choloyl/chenodeoxycholoyl-CoA",dTDP-D-angolosamine biosynthesis,dTDP-D-desosamine biosynthesis,dTDP-D-forosamine biosynthesis,dTDP-L-megosamine biosynthesis,dTDP-L-olivose biosynthesis,dTDP-L-rhamnose biosynthesis
0,0.0,52.78,0.0,0.0,0.0,0.0,0.0,100.0,77.78,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.0
1,0.0,22.22,0.0,100.0,0.0,0.0,0.0,100.0,100.00,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,50.0
2,0.0,11.11,0.0,100.0,0.0,0.0,0.0,100.0,66.67,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.0
3,0.0,11.11,0.0,100.0,0.0,0.0,0.0,100.0,66.67,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.0
4,0.0,27.78,0.0,20.0,0.0,0.0,0.0,100.0,100.00,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,0.0,25.00,0.0,0.0,0.0,0.0,50.0,100.0,66.67,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.0
186,0.0,25.00,0.0,0.0,0.0,0.0,50.0,100.0,66.67,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.0
187,0.0,27.78,0.0,0.0,20.0,0.0,0.0,100.0,50.00,0.0,...,100.0,100.0,33.33,0.0,0.0,0.0,0.0,0.0,0.0,100.0
188,0.0,24.54,0.0,0.0,20.0,0.0,0.0,100.0,50.00,0.0,...,50.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,50.0


## MQ: Medium-quality species-level clusters (contamination < 10% and completeness > 50%)


In [13]:
mq_df = reps_df.query("Contamination < 10 and Completeness > 50")
compute_print_stats(mq_df)
print(f"Total number: {len(mq_df)} ({len(mq_df) / len(reps_df) * 100:.2f}%)")

Total number: 328.0
Cluster members: 1.16 ± 0.43, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-3.00
Contamination: 2.17 ± 2.08, Median: 1.61, IQR: 0.78-2.86, Range: 0.00-9.98
Completeness: 86.59 ± 14.03, Median: 92.25, IQR: 80.15-97.80, Range: 50.60-100.00
Total length: 3.37 ± 1.23, Median: 3.19, IQR: 2.47-4.02, Range: 0.22-7.12
Total number: 328 (39.42%)


### Taxonomy

In [14]:
compute_taxo_classification_summary(mq_df)

,Unclassified clusters,Classified clusters,Unclassified clusters %,Classified clusters %
Domain,2,326,0.61,99.39
Phylum,2,326,0.61,99.39
Class,2,326,0.61,99.39
Order,2,326,0.61,99.39
Family,8,320,2.44,97.56
Genus,40,288,12.2,87.8
Species,300,28,91.46,8.54


In [15]:
mq_taxo_levels = get_all_taxo_levels(mq_df)


Level: Domain


,Cluster,Cluster %,Total MAG count
Domain,,,
Bacteria,323.0,98.475610,373.0
unclassified,2.0,0.609756,3.0
Archaea,2.0,0.609756,2.0
Pseudomonadati,1.0,0.304878,2.0
TOTAL,328.0,100.000000,380.0



Level: Phylum


,Cluster,Cluster %,Total MAG count
Phylum,,,
Pseudomonadota,166.0,50.609756,175.0
Bacteroidota,74.0,22.560976,106.0
Actinomycetota,20.0,6.097561,24.0
Cyanobacteriota,16.0,4.878049,16.0
Myxococcota_A,10.0,3.048780,10.0
Verrucomicrobiota,10.0,3.048780,10.0
Bdellovibrionota,6.0,1.829268,8.0
Patescibacteriota,6.0,1.829268,8.0
Planctomycetota,6.0,1.829268,6.0



Level: Class


...

,Cluster,Cluster %,Total MAG count
Family,,,
Flavobacteriaceae,66.0,20.121951,96.0
Maricaulaceae,19.0,5.792683,19.0
Rhodobacteraceae,18.0,5.487805,18.0
Granulosicoccaceae,16.0,4.878049,16.0
Cellvibrionaceae,14.0,4.268293,15.0
Alteromonadaceae,10.0,3.048780,15.0
Sphingomonadaceae,15.0,4.573171,15.0
Arenicellaceae,13.0,3.963415,13.0
Rhizobiaceae,13.0,3.963415,13.0



Level: Genus


,Cluster,Cluster %,Total MAG count
Genus,,,
unclassified,40.0,12.195122,43.0
Aquimarina,13.0,3.963415,15.0
Arenicella,13.0,3.963415,13.0
JAALLB01,12.0,3.658537,12.0
SHLQ01,8.0,2.439024,12.0
...,...,...,...
Maribacter_B,1.0,0.304878,1.0
Litoreibacter,1.0,0.304878,1.0
Lentilitoribacter,1.0,0.304878,1.0



Level: Species


,Cluster,Cluster %,Total MAG count
Species,,,
unclassified,300.0,91.463415,335.0
Olleya sediminilitoris,1.0,0.304878,3.0
Cellulophaga lytica,1.0,0.304878,3.0
Dokdonia sp947496725,1.0,0.304878,3.0
Pseudoalteromonas marina,1.0,0.304878,3.0
Polaribacter marinaquae,1.0,0.304878,2.0
Croceitalea vernalis,1.0,0.304878,2.0
Lacinutrix sp000211855,1.0,0.304878,2.0
Maribacter litoralis,1.0,0.304878,2.0


### Relative abundance

In [16]:
mq_relative_abund_df = get_relative_abund_taxo_levels(mq_df, coverage_df)

Unmapped reads: 73.26 ± 8.53, Median: 69.14, IQR: 68.35-76.10, Range: 67.56-83.06
Mapped reads: 26.74 ± 8.53, Median: 30.86, IQR: 23.90-31.65, Range: 16.94-32.44

Level: Family


,count,mean,std,min,25%,50%,75%,max
Family,,,,,,,,
Flavobacteriaceae,3.0,42.058922,10.065736,35.192530,36.281580,37.370631,45.492118,53.613604
Cellvibrionaceae,3.0,7.934858,10.126733,0.490720,2.169040,3.847360,11.656927,19.466494
Sphingomonadaceae,3.0,4.333819,3.692108,0.178980,2.881392,5.583804,6.411239,7.238674
Maricaulaceae,3.0,4.004491,3.044146,0.677865,2.681154,4.684443,5.667804,6.651165
Alteromonadaceae,3.0,3.599345,1.992512,1.458229,2.699435,3.940640,4.669902,5.399165
Arenicellaceae,3.0,3.418590,1.365231,1.938442,2.813646,3.688850,4.158665,4.628479
Vibrionaceae,3.0,2.960235,4.228998,0.066327,0.533570,1.000813,4.407189,7.813565
Granulosicoccaceae,3.0,2.521925,1.468668,0.914845,1.885657,2.856468,3.325465,3.794461
Rhodobacteraceae,3.0,2.476063,2.406922,0.904243,1.090597,1.276951,3.261973,5.246994



Level: Genus


,count,mean,std,min,25%,50%,75%,max
Genus,,,,,,,,
unclassified,3.0,13.626611,13.955864,5.406590,5.569740,5.732889,17.736622,29.740354
Aquimarina,3.0,12.526848,18.888875,0.747715,1.633344,2.518973,18.416414,34.313854
Cellulophaga,3.0,7.038323,9.846555,0.242826,1.392279,2.541733,10.436071,18.330409
Postechiella,3.0,5.244876,5.443666,0.516750,2.269306,4.021862,7.608938,11.196015
Arenicella,3.0,3.418590,1.365231,1.938442,2.813646,3.688850,4.158665,4.628479
...,...,...,...,...,...,...,...,...
Lentilitoribacter,3.0,0.103203,0.113676,0.019636,0.038480,0.057324,0.144986,0.232648
CALGNO01,3.0,0.091942,0.159249,0.000000,0.000000,0.000000,0.137914,0.275827
Roseibacillus,3.0,0.087884,0.152220,0.000000,0.000000,0.000000,0.131826,0.263653



Level: Species


,count,mean,std,min,25%,50%,75%,max
Species,,,,,,,,
unclassified,3.0,79.521767,13.879605,66.218567,72.325822,78.433078,86.173367,93.913657
Cellulophaga lytica,3.0,7.038323,9.846555,0.242826,1.392279,2.541733,10.436071,18.330409
Dokdonia sp947496725,3.0,2.940641,3.399557,0.811150,0.980338,1.149525,4.005386,6.861247
Vibrio cyclitrophicus,3.0,1.179877,1.696689,0.024332,0.205920,0.387507,1.757650,3.127792
Tenacibaculum sp004337695,3.0,0.938796,0.850838,0.239604,0.465137,0.690670,1.288393,1.886115
Vibrio coralliirubri,3.0,0.916185,1.264004,0.017500,0.193522,0.369544,1.365527,2.361510
Pseudoalteromonas atlantica,3.0,0.912028,0.747816,0.127268,0.559839,0.992410,1.304408,1.616406
Lacinutrix sp000211855,3.0,0.882009,0.706106,0.104280,0.581583,1.058885,1.270873,1.482861
unclassified Vibrio,3.0,0.864173,1.269219,0.024496,0.134128,0.243761,1.284012,2.324263


### Functions

In [17]:
print_stats(get_bakta_annot_df(mq_df).describe())

CDSs: 3084.15 ± 1038.63, Median: 2975.00, IQR: 2299.50-3613.00, Range: 370.00-5968.00
CRISPR arrays: 0.12 ± 0.58, Median: 0.00, IQR: 0.00-0.00, Range: 0.00-4.00
gaps: 0.00 ± 0.00, Median: 0.00, IQR: 0.00-0.00, Range: 0.00-0.00
hypotheticals: 1049.28 ± 631.71, Median: 842.00, IQR: 559.00-1459.50, Range: 84.00-3018.00
ncRNA regions: 5.68 ± 3.21, Median: 5.00, IQR: 3.00-8.00, Range: 0.00-15.00
ncRNAs: 5.36 ± 5.74, Median: 4.00, IQR: 3.00-6.00, Range: 0.00-51.00
oriCs: 0.02 ± 0.16, Median: 0.00, IQR: 0.00-0.00, Range: 0.00-2.00
oriTs: 0.00 ± 0.00, Median: 0.00, IQR: 0.00-0.00, Range: 0.00-0.00
oriVs: 0.00 ± 0.00, Median: 0.00, IQR: 0.00-0.00, Range: 0.00-0.00
pseudogenes: 6.74 ± 7.74, Median: 4.00, IQR: 1.00-9.00, Range: 0.00-40.00
rRNAs: 0.84 ± 1.07, Median: 1.00, IQR: 0.00-1.00, Range: 0.00-8.00
sORFs: 0.19 ± 0.44, Median: 0.00, IQR: 0.00-0.00, Range: 0.00-2.00
signal peptides: 0.00 ± 0.00, Median: 0.00, IQR: 0.00-0.00, Range: 0.00-0.00
tRNAs: 32.41 ± 11.42, Median: 31.00, IQR: 25.00-38.

In [18]:
get_kegg_path_df(mq_df)

Before removing rows and columns with only zeros:
Clusters: 328
KEGG modules: 405

After removing rows and columns with only zeros:
Clusters: 328
KEGG modules: 386

KEGG modules: 188.32 ± 30.16, Median: 192.00, IQR: 181.00-201.25, Range: 2.00-245.00


,"10-membered enediyne core biosynthesis, malonyl-CoA => 3-hydroxyhexadeca-4,6,8,10,12,14-hexaenoyl-ACP => 10-membered enediyne core","2-Oxocarboxylic acid chain extension, 2-oxoglutarate => 2-oxoadipate => 2-oxopimelate => 2-oxosuberate",3-Hydroxypropionate bi-cycle,"9-membered enediyne core biosynthesis, malonyl-CoA => 3-hydroxyhexadeca-4,6,8,10,12,14-hexaenoyl-ACP => 9-membered enediyne core",ADP-L-glycero-D-manno-heptose biosynthesis,"Abscisic acid biosynthesis, beta-carotene => abscisic acid","Acarbose biosynthesis, sedoheptulopyranose-7P => acarbose",Acylglycerol degradation,"Adenine ribonucleotide biosynthesis, IMP => ADP,ATP","Adenine ribonucleotide degradation, AMP => Urate",...,"beta-Oxidation, peroxisome, tri/dihydroxycholestanoyl-CoA => choloyl/chenodeoxycholoyl-CoA",dTDP-D-angolosamine biosynthesis,dTDP-D-desosamine biosynthesis,dTDP-D-forosamine biosynthesis,dTDP-D-mycaminose biosynthesis,dTDP-L-megosamine biosynthesis,dTDP-L-mycarose biosynthesis,dTDP-L-olivose biosynthesis,dTDP-L-rhamnose biosynthesis,dTDP-beta-L-noviose biosynthesis
0,0.0,0.0,52.78,0.0,0.0,0.0,0.0,0.0,100.0,77.78,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0
1,0.0,0.0,22.22,0.0,100.0,0.0,0.0,0.0,100.0,100.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,50.00,0.0
2,0.0,0.0,11.11,0.0,100.0,0.0,0.0,0.0,100.0,66.67,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0
3,0.0,0.0,11.11,0.0,100.0,0.0,0.0,0.0,100.0,66.67,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0
4,0.0,0.0,27.78,0.0,20.0,0.0,0.0,0.0,100.0,100.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,0.0,0.0,13.89,0.0,0.0,0.0,0.0,0.0,50.0,66.67,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,50.00,0.0
326,0.0,0.0,13.89,0.0,0.0,0.0,0.0,0.0,50.0,66.67,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,50.00,0.0
327,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0
328,0.0,0.0,14.81,0.0,0.0,0.0,12.5,0.0,75.0,33.33,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,33.33,0.0


## LQ: Low-quality species-level clusters (contamination < 10% and completeness < 50%)

In [19]:
lq_df = reps_df.query("Contamination < 10 and Completeness < 50")
compute_print_stats(lq_df)
print(f"Total number: {len(lq_df)} ({len(lq_df) / len(reps_df) * 100:.2f}%)")

Total number: 502.0
Cluster members: 1.03 ± 0.16, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-2.00
Contamination: 0.89 ± 1.60, Median: 0.17, IQR: 0.01-1.01, Range: 0.00-8.14
Completeness: 16.72 ± 12.34, Median: 11.55, IQR: 7.40-24.30, Range: 0.00-49.40
Total length: 0.80 ± 0.82, Median: 0.54, IQR: 0.29-1.06, Range: 0.05-7.59
Total number: 502 (60.34%)


## Non classified

In [20]:
nc_df = reps_df.query("Contamination >= 10")
print(f"Total number: {len(nc_df)} ({len(nc_df) / len(reps_df) * 100:.2f}%)")

Total number: 2 (0.24%)
